<a href="https://colab.research.google.com/github/maxkolos/fin_llm/blob/llm_workspace/llm_workspace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Structure



1.   [How to Run](#scrollTo=I2j4hOKTOmVJ): This section covers the manual Google Colab UI steps required to configure your hardware accelerator prior to executing any cell.
2.  [Initialization Workflow](#scrollTo=ZOKy4yesmDHD): This section handles the automated, one-time setup prior to prompting an LLM.
3.  [Workspace](#scrollTo=9sb17CTCQuRM): This section provides the active workspace for your custom prompts to LLM.



# How to Run

This section covers the manual Google Colab UI steps required to configure your hardware accelerator prior to executing any code.

1. Click the **Runtime** drop-down menu in the upper-right corner and select **Change runtime type**.

![Runtime settings](https://raw.githubusercontent.com/maxkolos/fin_llm/refs/heads/main/images/runtime_settings.png)



2. Select **T4 GPU** from the Hardware accelerator list and click **Save**.

![Runtime type](https://raw.githubusercontent.com/maxkolos/fin_llm/refs/heads/main/images/runtime_type.png)



3. In the top menu bar, select **Run all**.

![Run all](https://raw.githubusercontent.com/maxkolos/fin_llm/refs/heads/main/images/run_all.png)


# Initialization Workflow

This section handles the automated, **one-time setup** prior to prompting an LLM.

In [ ]:
# @title Install Python packages
!pip uninstall -qq -y torch torchvision torchaudio

!pip install torch -qq  --index-url https://pytorch.org
!pip install vllm -qq --extra-index-url https://pytorch.org

# Apply changes without a session restart.
# IMPORTANT: don't import torch or vllm before this point (e.g. 'import torch').
import site
site.main()

print("SUCCESS: All packages have been installed.")

In [ ]:
# @title Test the installed packages and GPU availability
import torch
import vllm

print("Does PyTorch see GPU?", torch.cuda.is_available())
print("CUDA version in PyTorch:", torch.version.cuda)
print("SUCCESS: vLLM and PyTorch are imported successfully!")

In [ ]:
# @title Download an LLM model
!pip install -q huggingface_hub hf_transfer

from huggingface_hub import snapshot_download
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Downloading Qwen2.5-14B-Instruct-AWQ...")
snapshot_download(repo_id="Qwen/Qwen2.5-14B-Instruct-AWQ",
                  max_workers=8)
print("\nSUCCESS: The model has been downloaded.")

In [ ]:
# @title Start a vLLM server
import fcntl
import os
import requests
import time
import subprocess
import sys

env = os.environ.copy()
env["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "0"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

command = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-14B-Instruct-AWQ",
    "--port", "8000",
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.82",
    "--kv-cache-dtype", "auto"
  ]

server_process = subprocess.Popen(
    command,
    # Redirect stderr and stdout in case logs are needed for debugging.
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env
)

# Make stdout non-blocking.
fd = server_process.stdout.fileno()
fl = fcntl.fcntl(fd, fcntl.F_GETFL)
fcntl.fcntl(fd, fcntl.F_SETFL, fl | os.O_NONBLOCK)

print("vLLM server is starting...\n")

while True:
    if server_process.poll() is not None:
        print("\n[FATAL] vLLM failed to start!")
        remaining_output = server_process.stdout.read()
        print(remaining_output)
        break

    # Non-blocking read of server's log.
    try:
        while True:
            line = server_process.stdout.readline()
            if not line:
                break
            print(line.strip())
    except IOError:
        pass

    # Check whether the server is ready.
    try:
        response = requests.get("http://localhost:8000/v1/models")
        if response.status_code == 200:
            print("\nSUCCESS: The server is ready!")
            break
    except requests.exceptions.ConnectionError:
        # print(".", end="", flush=True)
        time.sleep(5)  # Wait longer.


In [ ]:
# @title Connect to the vLLM server
!pip install -q openai

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="token-not-needed"
)

print("SUCCESS: A server connection is established.")

In [ ]:
%%time
# @title Build a multi-agent system
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Shared State of Agents
class AgentState(TypedDict):
    news: str          # Input market news
    analysis: str      # Analyst recommendation
    feedback : str     # Critic feedback
    iterations: int    # Validation loop counter

def call_local_llm(prompt: str) -> str:
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-14B-Instruct-AWQ",
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=1000
    )
    return response.choices[0].message.content

# Node 1: Analyst (Generates and refines recommendation)
def analyst_node(state: AgentState):
    iterations = state.get("iterations") + 1

    prompt = f"Analyze these stock market news and give a buy/sell recommendation:\n{state['news']}"
    if "feedback" in  state:
        prompt += (
            f"\n\nTake into account critic's feedback to your previous analysis.\n\n"
            f"Your previous analysis: {state['analysis']}\n\n"
            f"Critique to address: {state['feedback']}\n\n"
        )
    response = call_local_llm(prompt)
    return {"analysis": response, "iterations": iterations}

# Node 2: Critic (Checks risks)
def critic_node(state: AgentState):
    prompt = f"Critique briefly this financial analysis for risks and missing facts:\n{state['analysis']}"
    response = call_local_llm(prompt)
    return {"feedback": response}

# Routing function (Exit condition)
def should_continue(state: AgentState):
    if state["iterations"] >= 2:
        return "end"
    return "continue"

# Graph assembly
workflow = StateGraph(AgentState)
# Add nodes
workflow.add_node("analyst", analyst_node)
workflow.add_node("critic", critic_node)
# Add edges
workflow.add_edge(START, "analyst")
workflow.add_conditional_edges(
    "analyst",
    should_continue,
    {
        "continue": "critic",
        "end": END
    }
)
workflow.add_edge("critic", "analyst")

app = workflow.compile()

print("Graph structure:")
from IPython.display import Image, display
display(Image(data=app.get_graph().draw_mermaid_png()))

# Workspace

This section provides the active workspace for your custom prompts to LLM.

In [ ]:
# @title Plain prompt {"single-column":true}
system_message = "You are an experienced AI system designer" # @param {"type":"string","placeholder":"Enter the role of AI agent"}
prompt = "Explain the advantages of on-premises deployment of an LLM compared to a cloud API" # @param {"type":"string","placeholder":"Enter your prompt"}

print("Sending a prompt to LLM...")

completion = client.chat.completions.create(
    model="Qwen/Qwen2.5-14B-Instruct-AWQ",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ],
    temperature=0.3,
    max_tokens=300
)

print("\n--- LLM response ---")
print(completion.choices[0].message.content)


In [ ]:
%%time
# @title Prompt to the multi-agent system {"single-column":true}
prompt = "Apple reports record Q3 earnings, but supply chain issues trigger 2% drop in after-hours trading." # @param {"type":"string","placeholder":"Enter your prompt"}

inputs = {
    "news": prompt,
    "iterations": 0
}

result = app.invoke(inputs)
print("\n--- Multi-agent system's response ---")
print(result["analysis"])